# Experimentos de Chunking — CLT

**Pessoa 1 — Felipe Teodoro**

Comparação entre duas estratégias de chunking para o corpus da CLT:
- **Estratégia A:** chunking por tamanho fixo (`RecursiveCharacterTextSplitter`)
- **Estratégia B:** chunking por artigo (unidade semântica natural da CLT) ← estratégia adotada

A estratégia B foi escolhida pois preserva a integridade dos artigos e permite incluir metadados (`artigo`, `secao`) em cada chunk, viabilizando citação da fonte nas respostas do chatbot.

In [3]:
import sys
sys.path.append('../src/ingestion')
from extractor import extract_text_from_pdf
from chunker import chunk_text, chunk_by_article

## 1. Extração do PDF

In [4]:
text = extract_text_from_pdf('../data/raw/clt.pdf')
print(f'Total de caracteres: {len(text):,}')
print(f'\nPrimeiros 500 caracteres:\n{text[:500]}')

Total de caracteres: 744,289

Primeiros 500 caracteres:
edições
câmara
Câmara dos
Deputados
CLT
CONSOLIDAÇÃO
DAS LEIS DO
TRABALHO
4ª EDIÇÃO
INCLUI
Lei do Trabalho Doméstico
Lei do Trabalho Rural
Câmara dos Deputados
57ª Legislatura | 2023 – 2027
Presidente
Arthur Lira
1º Vice-Presidente
Marcos Pereira
2º Vice-Presidente
Sóstenes Cavalcante
1º Secretário
Luciano Bivar
2ª Secretária
Maria do Rosário
3º Secretário
Júlio Cesar
4º Secretário
Lucio Mosquini
Suplentes de secretários
1º Suplente
Gilberto Nascimento
2º Suplente
Pompeo de Mattos
3º Suplente
Be


## 2. Estratégia A — Chunking por Tamanho Fixo

Divide o texto a cada N caracteres com overlap, sem respeitar fronteiras semânticas.
Problema: um artigo pode ser cortado ao meio, ou dois artigos diferentes ficam no mesmo chunk.

In [5]:
configs = [
    {'chunk_size': 500,  'chunk_overlap': 50},
    {'chunk_size': 1000, 'chunk_overlap': 200},
    {'chunk_size': 1500, 'chunk_overlap': 300},
]

print(f"{'Tamanho':>8} {'Overlap':>8} {'Chunks':>8} {'Média (chars)':>15}")
print("-" * 45)
for cfg in configs:
    chunks = chunk_text(text, **cfg)
    avg_len = sum(len(c) for c in chunks) / len(chunks)
    print(f"{cfg['chunk_size']:>8} {cfg['chunk_overlap']:>8} {len(chunks):>8} {avg_len:>15.0f}")

 Tamanho  Overlap   Chunks   Média (chars)
---------------------------------------------
     500       50     1653             455
    1000      200      936             955
    1500      300      620            1456


## 3. Problema da Estratégia A — Exemplo Visual

Mostra que um chunk pode misturar dois artigos diferentes.

In [6]:
import re

chunks_fixo = chunk_text(text, chunk_size=1000, chunk_overlap=200)

print("=== Chunks que contêm mais de um artigo (Estratégia A) ===\n")
multi_artigo = [c for c in chunks_fixo if len(re.findall(r'Art\.\s+\d+', c)) > 1]
print(f"Chunks com múltiplos artigos: {len(multi_artigo)} de {len(chunks_fixo)} ({len(multi_artigo)/len(chunks_fixo)*100:.1f}%)")
print(f"\nExemplo de chunk com dois artigos:\n")
print(multi_artigo[0][:600], "...")

=== Chunks que contêm mais de um artigo (Estratégia A) ===

Chunks com múltiplos artigos: 359 de 936 (38.4%)

Exemplo de chunk com dois artigos:

cargo de direção ou representação sindical e, se eleito, ainda que suplente, até um ano após o 
final do mandato, salvo se cometer falta grave nos termos da lei.
Parágrafo único. As disposições deste artigo aplicam-se à organização de sindicatos rurais e 
de colônias de pescadores, atendidas as condições que a lei estabelecer.
Art. 9º É assegurado o direito de greve, competindo aos trabalhadores decidir sobre a oportu -
nidade de exercê-lo e sobre os interesses que devam por meio dele defender.
§ 1º A lei definirá os serviços ou atividades essenciais e disporá sobre o atendimento das 
necessid ...


## 4. Estratégia B — Chunking por Artigo (Adotada)

Divide o texto usando `Art. X` como delimitador. Cada chunk corresponde a um artigo completo.
Artigos muito longos são subdivididos com overlap de 150 chars, mantendo o número do artigo nos metadados.

### Campos de metadados disponíveis no ChromaDB

| Campo | Tipo | Exemplo | Descrição |
|-------|------|---------|-----------|
| `artigo` | string | `"Art. 130"` | Número do artigo da CLT |
| `secao` | string | `"TÍTULO IV – Do Contrato Individual de Trabalho"` | Título/Capítulo/Seção em que o artigo está inserido |

In [7]:
chunks_artigo = chunk_by_article(text)

tamanhos = [len(c['text']) for c in chunks_artigo]
print(f"Total de chunks : {len(chunks_artigo)}")
print(f"Média de chars  : {sum(tamanhos)/len(tamanhos):.0f}")
print(f"Menor chunk     : {min(tamanhos)} chars")
print(f"Maior chunk     : {max(tamanhos)} chars")

print(f"\nChunks com múltiplos artigos: 0 (por definição da estratégia)")

Total de chunks : 1197
Média de chars  : 636
Menor chunk     : 50 chars
Maior chunk     : 1500 chars

Chunks com múltiplos artigos: 0 (por definição da estratégia)


In [8]:
# Inspeciona 3 chunks com seus metadados
for chunk in chunks_artigo[10:13]:
    print(f"artigo : {chunk['artigo']}")
    print(f"secao  : {chunk['secao']}")
    print(f"texto  : {chunk['text'][:300]}")
    print("-" * 60)

artigo : Preâmbulo
secao  : Título I – Introdução ................................................................................................................................14
texto  : paro dos trabalhadores, que eram submetidos a longas e exaustivas jornadas de trabalho, 
em instalações precárias, sem a observância de cuidados com a integridade física e mental. 
Com a edição da CL T , buscou-se consolidar um regramento que conferisse direitos aos 
trabalhadores.
Por intermédio da
------------------------------------------------------------
artigo : Preâmbulo
secao  : Título I – Introdução ................................................................................................................................14
texto  : ticos não eram assegurados os mesmos direitos garantidos aos demais empregados brasi-
leiros, fato esse que, aos olhos de muitos, caracterizava o trabalhador doméstico como um 
“trabalhador de segunda classe” . Por esse motivo, podemos dizer que a CL T foi d

## 5. Como usar os metadados na recuperação (Pessoa 2)

Ao recuperar documentos do ChromaDB, cada resultado vem com `.metadata`:

```python
docs = retriever.invoke("férias anuais remuneradas")
for doc in docs:
    print(doc.metadata["artigo"])   # "Art. 130"
    print(doc.metadata["secao"])    # "TÍTULO IV..."
    print(doc.page_content)         # texto do chunk
```

Para exibir a fonte na resposta do chatbot, inclua os metadados no prompt:

```python
contexto = "\n\n".join(
    f"[{doc.metadata['artigo']}] {doc.page_content}"
    for doc in docs
)
```

## 6. Conclusão

| | Estratégia A (tamanho fixo) | Estratégia B (por artigo) |
|---|---|---|
| Chunks | ~936 | ~1197 |
| Integridade semântica | Baixa — artigos cortados | Alta — um artigo por chunk |
| Metadados | Nenhum | `artigo`, `secao` |
| Citação da fonte | Impossível | `"Conforme Art. 130 da CLT..."` |
| Artigos subdivididos | N/A | Sim, com overlap de 150 chars |

**Decisão:** Estratégia B adotada. O vectorstore commitado em `data/vectorstore/` foi gerado com `chunk_by_article()` usando `MAX_CHUNK_SIZE=1500`.